# 07 · From tracks to events and statistics

`analytics.run_analytics`, unrolled. Input: `tracks` (people in pitch metres, team, role) + `ball` (clean trajectory).
Uses the full-match run of `params.yaml → full_run_dir` when it exists (events need a whole match), else the window
of 06. The rules and thresholds are documented in `docs/STATS_RULEBOOK.md`; each cell maps to one of them.

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "football_eval").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from football_eval import nbtools as nbt

P = nbt.load_params()            # notebooks/pipeline/params.yaml (or $FVE_NB_PARAMS)
ctx = nbt.build_context(P)       # config <- match setup, restricted to the window
cfg = ctx.cfg

In [ ]:
from football_eval.perception import load_perception
from football_eval.analytics import restrict_to_periods, run_analytics
from football_eval.context import infer_periods, player_speeds
from football_eval.teams import assign_goalkeepers
from football_eval.pitch import Pitch, draw_pitch
from football_eval.possession import frame_control, touches, team_possession
from football_eval.events import EventDetector, dead_ball
from football_eval.xg import XGModel
from football_eval.stats import team_stats, player_stats, possession_timeline
from football_eval.timeline import build_timeline
src = Path(P.full_run_dir) if P.full_run_dir else None
src = (ROOT / src) if src is not None and not src.is_absolute() else src
if src is None or not (src / "tracks.csv").exists():
    src = ctx.out / "perception"
print("analytics input:", src)
res = load_perception(str(src))
tracks, ball = res["tracks"], res["ball"]
print(f"{len(tracks)} person rows, {tracks.track_id.nunique()} track ids, {len(ball)} ball rows "
      f"({ball.t.min() / 60:.1f}-{ball.t.max() / 60:.1f} min), ball usable {ball.valid.mean():.0%}")

## 1 · Periods, playing directions, goalkeepers

In [ ]:
pitch = Pitch(cfg.pitch.length, cfg.pitch.width)
if cfg.match.halves and cfg.video.only_periods:
    tracks, ball, cut = restrict_to_periods(tracks, ball, cfg.match.halves)
    print(f"outside the periods: {cut} frames dropped")
mctx = infer_periods(tracks, pitch, cfg.match.halves, cfg.match.home_attacks_right_first_half)
mctx.team_names = list(cfg.teams.team_names)
if (tracks.cls == "goalkeeper").sum() == 0:
    for i, (a, b) in enumerate(mctx.periods):
        sel = (tracks.t >= a) & (tracks.t <= b)
        if sel.any():
            sub = assign_goalkeepers(tracks[sel].copy(), pitch, dict(mctx.attacks_right[i]))
            tracks.loc[sel, ["team", "cls"]] = sub[["team", "cls"]].to_numpy()
tracks["team"] = tracks.team.astype(int)
print("periods (video s):", mctx.periods, "| attacks right:", mctx.attacks_right)
tr_sp = player_speeds(tracks)
tr_sp.groupby("team").speed.describe()[["mean", "50%", "max"]]

## 2 · Ball control per frame → dead ball → touches → possession

In [ ]:
fc = frame_control(tracks, ball, cfg)
dead, stoppages = dead_ball(fc, tr_sp, mctx, cfg)
fc.loc[dead, ["ctrl_tid", "ctrl_team"]] = -1
tch = touches(fc, tracks, cfg.possession.min_control_s)
poss = team_possession(fc, dead, cfg.possession.team_switch_s)
if "cam_ok" in fc:
    unknown = ~fc.cam_ok.fillna(True).astype(bool).to_numpy(); poss[unknown] = -1; dead = dead & ~unknown
live = ~dead
print(f"frames: {len(fc)} | dead ball {dead.mean():.0%} | {len(stoppages)} stoppages | {len(tch)} touches")
print("possession (live frames with a team):", {mctx.team_names[k]: f"{(poss[live] == k).sum() / max((poss[live] >= 0).sum(), 1):.0%}" for k in (0, 1)},
      f"| unknown {(poss[live] < 0).mean():.0%}")
pt = possession_timeline(fc, poss, dead)
pt.head()

## 3 · Events and xG

In [ ]:
xg_path = ROOT / cfg.xg.model_file
xgm = XGModel.load(xg_path) if xg_path.exists() else None
ev = EventDetector(cfg, mctx, xgm).run(fc, tch, tracks, stoppages, res.get("meta", {}).get("scoreboard_changes", []))
ev["clock"] = [mctx.match_clock(t) for t in ev.t]
print(ev.type.value_counts().to_dict())
ev.head(12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
draw_pitch(axes[0], pitch)
for k, c in ((0, "C3"), (1, "C0")):
    p_ = ev[(ev.type == "pass") & (ev.team == k)]
    axes[0].quiver(p_.x, p_.y, p_.x_end - p_.x, p_.y_end - p_.y, angles="xy", scale_units="xy", scale=1, color=c, width=0.002, alpha=0.6, label=mctx.team_names[k])
axes[0].legend(fontsize=8); axes[0].set_title("passes")
draw_pitch(axes[1], pitch)
sh = ev[ev.type == "shot"]
if len(sh):
    axes[1].scatter(sh.x, sh.y, s=800 * sh.xg.fillna(0.02) + 20, c=["C3" if t == 0 else "C0" for t in sh.team], alpha=0.7)
axes[1].set_title(f"shots ({len(sh)}), size = xG");

## 4 · Statistics and the timeline

In [ ]:
tstats = team_stats(ev, fc, poss, dead, tr_sp, mctx, cfg)
pstats = player_stats(ev, tr_sp, tch)
tl = build_timeline(ev, mctx, final_third_x=cfg.events.final_third_x)
tstats

In [ ]:
pstats.sort_values(pstats.columns[-1], ascending=False).head(10) if len(pstats) else pstats

## 5 · Cross-check with `run_analytics` (and the match sheet)
The library run also applies the match sheet / review verdicts and the pressure phases, so it can have *more* events; the automatic counts should match the cells above.

In [ ]:
full = run_analytics(res["tracks"], res["ball"], cfg, res.get("meta", {}).get("scoreboard_changes", []), root=str(ROOT), setup=ctx.setup)
c = pd.DataFrame({"unrolled (this notebook)": ev.type.value_counts(), "run_analytics: automatic": full["events_auto"].type.value_counts(),
                  "run_analytics: final (setup / review applied)": full["events"].type.value_counts()}).fillna(0).astype(int)
if ctx.setup:
    print("match sheet:", [(e.get("type"), e.get("minute"), e.get("team")) for e in ctx.setup.get("known_events", [])])
c